# 15 · Train, review and launch the teacher-trained AHR player

**Purpose:** prepare validated 9×9 teacher examples, pretrain AHR Scout/Worker networks, and review pinned candidates against GNU Go. **Inputs:** the immutable `quintic/go9x9` release and notebooks 00–03, 05, 08, 12–14. **Outputs:** resumable checkpoints in `checkpoints/<run_id>/`, reports under `reports/ahr_v2/<run_id>/`, and an active-player pointer consumed by notebook 16.

**Runtime:** Run All performs bounded validation and presents status. The explicit launch function starts preparation and up to eight hours of training in detached tmux. Preparation and independent match reviews add wall time. Training may finish earlier at its configured epoch/step limit. **Exact resume:** `go2_launch_teacher(PROJECT, resume=True)` using the saved configuration and the same notebook exports. Never modify a running run's pinned source/config/data identity. Attach with `tmux attach -t ahr-v2-teacher`; detach with Ctrl-b, then d.

External KataGo estimates are training supervision, not proofs. Local group readings remain bounded; a cutoff stays UNKNOWN. Development matches are not training data. The dataset's test split remains untouched. Storage is the persistent boot disk; off-VM backup was deferred by the user.

In [ ]:
from pathlib import Path
import nbformat
PROJECT=Path.cwd().resolve()
while not (PROJECT/'AGENTS.md').is_file() and PROJECT!=PROJECT.parent: PROJECT=PROJECT.parent
for cell in nbformat.read(PROJECT/'experiments/ahr_v2/00_contracts_and_recovery.ipynb',4).cells:
    if cell.cell_type=='code' and 'ahr-export' in cell.metadata.get('tags',[]): exec(compile(cell.source,'00:ahr-export','exec'),globals())


## Independent matches with recoverable game and search state

In [ ]:
import collections, dataclasses, selectors, shutil, subprocess, shlex

GO2_TEACHER_FILES=('00_contracts_and_recovery.ipynb','01_rules_and_replay.ipynb','02_scout_worker_network.ipynb','03_batched_puct_search.ipynb','05_evaluation_and_progress.ipynb','08_tactical_constraints_and_search.ipynb','12_teacher_dataset_and_targets.ipynb','13_teacher_pretraining_and_recovery.ipynb','14_worker_reading_and_search.ipynb','15_teacher_evaluation_and_launch.ipynb','16_play_teacher_trained_player.ipynb')

def go2_teacher_control_sources(project):
    project=go2_project(project)
    return {GO2_ROOT+'/'+name:go2_code_sha(project/GO2_ROOT/name) for name in GO2_TEACHER_FILES}

def go2_teacher_arena_namespace(project=None):
    project=go2_project(project);namespace={}
    for name in GO2_TEACHER_FILES:
        if name.startswith(('15_','16_')):continue
        go2_import_exports(project/GO2_ROOT/name,namespace)
    namespace['go2_load_player']=namespace['go2_load_teacher_player']
    base_config=namespace['Go2EvalConfig']
    def validate_config(self):
        base_config.__post_init__(self)
        if not 1<=self.gnu_level<=10:raise ValueError('GNU Go level must be 1..10.')
    extended=dataclasses.make_dataclass('Go2TeacherArenaConfig',[('gnu_level',int,dataclasses.field(default=1))],bases=(base_config,),frozen=True,namespace={'__post_init__':validate_config})
    namespace['Go2EvalConfig']=extended
    base_gtp=namespace['Go2GTP']
    class TeacherGTP(base_gtp):
        def __init__(self,config,seed,executable=None):
            self.config=config
            self.path=Path(executable or shutil.which('gnugo') or '/usr/games/gnugo').resolve()
            if not self.path.is_file():raise FileNotFoundError('GNU Go dependency missing.')
            self.flags=['--mode','gtp','--level',str(config.gnu_level),'--seed',str(seed),'--chinese-rules','--positional-superko','--forbid-suicide','--never-resign','--capture-all-dead']
            self.process=subprocess.Popen([str(self.path),*self.flags],stdin=subprocess.PIPE,stdout=subprocess.PIPE,stderr=subprocess.DEVNULL,bufsize=0)
            self.selector=selectors.DefaultSelector();self.selector.register(self.process.stdout,selectors.EVENT_READ)
            self.buffer=bytearray();self.sequence=0
            try:
                self.version=self.command('version');self.command('boardsize '+str(config.board_size));self.command('clear_board');self.command('komi '+str(config.komi))
            except BaseException:self.close();raise
    namespace['Go2GTP']=TeacherGTP
    original_pin=namespace['go2_opponent_pin']
    def opponent_pin(config):
        pin=original_pin(config);pin['level']=config.gnu_level
        return pin
    namespace['go2_opponent_pin']=opponent_pin
    namespace['go2_eval_sources']=lambda project:go2_teacher_control_sources(project)
    return namespace

def go2_teacher_arena(project=None,run_id=None,eval_id=None,config=None,which='best',device='cuda:0',namespace=None,**fixture):
    project=go2_project(project);namespace=namespace or go2_teacher_arena_namespace(project)
    config=config or {'games':8,'simulations':128,'batch_size':8,'seed':2026100815,'max_plies':600,'game_time_seconds':900.0,'move_time_seconds':None,'gnu_level':1,'purpose':'development'}
    original_save=namespace['go2_save_checkpoint']
    cursor={'time':time.monotonic(),'step':None,'game':None}
    exact_fixture=fixture.get('_max_events') is not None or config.get('purpose')=='validation_fixture' if isinstance(config,dict) else config.purpose=='validation_fixture'
    def save_at_boundary(directory,state,best=False,resume_cell=None):
        step=state['global_step'];game=state.get('sampler',{}).get('game_index')
        due=exact_fixture or cursor['step'] is None or best or game!=cursor['game'] or time.monotonic()-cursor['time']>=120 or step-cursor['step']>=250
        if due:
            result=original_save(directory,state,best=best,resume_cell=resume_cell or '15: resume the same evaluation ID')
            cursor.update(time=time.monotonic(),step=step,game=game)
            return result
        return {'deferred_until_boundary':True,'global_step':step}
    namespace['go2_save_checkpoint']=save_at_boundary
    try:
        result=namespace['go2_evaluate'](project,run_id=run_id,eval_id=eval_id,config=config,which=which,device=device,**fixture)
    finally:namespace['go2_save_checkpoint']=original_save
    completed=[game for game in result['games'] if game['outcome']!='UNKNOWN']
    margins=[game['score_for_ahr'] for game in completed]
    result['diagnostics']={'mean_margin':float(np.mean(margins)) if margins else None,'unique_trajectories':len({go2_digest([item['action'] for item in game['trace']]) for game in result['games']}),'all_AHR_stones_gone':sum(not any(stone==game['ahr_color'] for stone in game['state']['board']) for game in completed)}
    go2_atomic_json(project/'reports/ahr_v2/evaluations'/result['eval_id']/'report.json',result)
    return result


## Validate exact recovery before any long run

In [ ]:
def go2_teacher_arena_validation(project=None):
    project=go2_project(project);namespace=go2_teacher_arena_namespace(project);torch.set_num_threads(2)
    token=uuid.uuid4().hex[:8];full_id='ahr-v2-teacher-arena-fixture-'+token+'-full';resume_id='ahr-v2-teacher-arena-fixture-'+token+'-resume'
    player=namespace['_go2_eval_fixture_player'](project)
    config={'games':2,'board_size':3,'simulations':4,'batch_size':2,'max_plies':8,'game_time_seconds':None,'purpose':'validation_fixture','gnu_level':1}
    fixture={'_player':player,'_opponent_factory':namespace['_Go2PassOpponent'],'_search_evaluator':namespace['_go2_pass_evaluator'](player[0])}
    full=go2_teacher_arena(project,eval_id=full_id,config=config,device='cpu',namespace=namespace,**fixture)
    partial=go2_teacher_arena(project,eval_id=resume_id,config=config,device='cpu',namespace=namespace,_max_events=2,**fixture)
    assert partial['status']=='PAUSED'
    saved,_=go2_load_checkpoint(project/'checkpoints'/resume_id);assert saved['search'] is not None
    path=project/'reports/ahr_v2/fixtures'/('teacher-arena-resume-'+token+'.ipynb');path.parent.mkdir(parents=True,exist_ok=True)
    lines=['from pathlib import Path','import nbformat','PROJECT=Path('+repr(str(project))+')',
      'for name in ["00_contracts_and_recovery.ipynb","15_teacher_evaluation_and_launch.ipynb"]:',
      '    for cell in nbformat.read(PROJECT/"experiments/ahr_v2"/name,4).cells:',
      '        if cell.cell_type=="code" and "ahr-export" in cell.metadata.get("tags",[]):exec(cell.source,globals())',
      'namespace=go2_teacher_arena_namespace(PROJECT)','torch.set_num_threads(2)','player=namespace["_go2_eval_fixture_player"](PROJECT)',
      'report=go2_teacher_arena(PROJECT,eval_id='+repr(resume_id)+',config='+repr(config)+',device="cpu",namespace=namespace,_opponent_factory=namespace["_Go2PassOpponent"],_search_evaluator=namespace["_go2_pass_evaluator"](player[0]))',
      'assert report["status"]=="COMPLETE"']
    book=nbformat.v4.new_notebook(metadata={'kernelspec':{'name':'ahr-go','language':'python','display_name':'AHR Go'}},cells=[nbformat.v4.new_code_cell(chr(10).join(lines))])
    go2_atomic_bytes(path,nbformat.writes(book).encode())
    subprocess.run([sys.executable,'-m','jupyter','nbconvert','--execute','--to','notebook','--inplace','--ExecutePreprocessor.kernel_name=ahr-go','--ExecutePreprocessor.timeout=240',str(path)],check=True,capture_output=True,text=True)
    after=json.loads((project/'reports/ahr_v2/evaluations'/resume_id/'report.json').read_text());assert full['games']==after['games'] and full['summary']==after['summary']
    counts=namespace['go2_eval_summary']([{'outcome':label,'ahr_color':1} for label in ('WIN','LOSS','DRAW','UNKNOWN')]);assert counts['unknown']==1 and counts['complete_games']==3
    gnu_config={'games':2,'board_size':3,'simulations':2,'batch_size':2,'max_plies':4,'game_time_seconds':60,'purpose':'validation_fixture','gnu_level':1}
    actual=go2_teacher_arena(project,eval_id='ahr-v2-teacher-gnu-smoke-'+token,config=gnu_config,device='cpu',namespace=namespace,_player=player)
    assert actual['summary']['unknown']==2
    for level in (5,10):
        cfg=namespace['Go2EvalConfig'](games=2,board_size=3,simulations=2,purpose='validation_fixture',gnu_level=level)
        with namespace['Go2GTP'](cfg,2026100815) as opponent:
            assert opponent.flags[3]==str(level) and opponent.version
        assert namespace['go2_opponent_pin'](cfg)['level']==level
    production=json.loads((project/'reports/ahr_v2/teacher_production_recovery.json').read_text())
    fixture_run=production['resumed_run_id']
    candidate=go2_teacher_preserve_candidate(project,fixture_run,_allow_smoke=True)
    model,metadata=go2_load_selected_teacher_player(project,fixture_run,device='cpu')
    original,original_metadata=namespace['go2_load_teacher_player'](project,fixture_run,device='cpu')
    assert metadata['is_smoke'] is True and go2_tensor_digest(model.state_dict())==go2_tensor_digest(original.state_dict())
    try:go2_teacher_preserve_candidate(project,fixture_run)
    except RuntimeError:pass
    else:raise AssertionError('A smoke candidate was allowed as a real playing result.')
    payload=go2_teacher_candidate_root(project,fixture_run)/candidate['file'];valid=payload.read_bytes()
    try:
        damaged=bytearray(valid);damaged[len(damaged)//2]^=1;go2_atomic_bytes(payload,bytes(damaged))
        try:go2_teacher_candidate_manifest(project,fixture_run)
        except RuntimeError:pass
        else:raise AssertionError('Candidate corruption was accepted.')
    finally:go2_atomic_bytes(payload,valid)
    receipt={'status':'PASS','checks':{'exact_pending_search_resume_in_fresh_kernel':True,'UNKNOWN_kept_separate':True,'real_GNU_Go_transport':True,'balanced_colors':True,'stage_or_120_seconds_or_250_events_checkpointing':True,'GNU_levels_5_and_10_transport_pinned':True,'full_candidate_copy_identity_and_smoke_provenance':True,'candidate_corruption_refused':True},'sources':go2_teacher_control_sources(project),'validated_at_utc':go2_now(),'fixture':str(path.relative_to(project))}
    go2_atomic_json(project/'reports/ahr_v2/teacher_arena_validation.json',receipt);return receipt


## Preserve playing candidates separately from the training best

In [ ]:
def go2_teacher_candidate_root(project,run_id):
    if Path(run_id).name!=run_id:raise ValueError('Run ID must be local.')
    return Path(project)/'checkpoints'/run_id/'playing_candidates'

def go2_teacher_candidate_manifest(project,run_id):
    root=go2_teacher_candidate_root(project,run_id);path=root/'manifest.json'
    if not path.exists():
        if list(root.glob('*.pt')):raise RuntimeError('Candidate payload without its manifest.')
        return None
    manifest=json.loads(path.read_text())
    if manifest.get('schema')!='ahr-v2-playing-candidates-v1' or not manifest.get('entries'):raise RuntimeError('Malformed playing-candidate manifest.')
    entries=manifest['entries'];names={entry['file'] for entry in entries}
    if len(names)!=len(entries) or manifest.get('best') not in names or manifest.get('latest')!=max(entries,key=lambda entry:entry['optimizer_step'])['file']:raise RuntimeError('Invalid playing-candidate cursor.')
    for entry in entries:
        if Path(entry['file']).name!=entry['file'] or not (root/entry['file']).is_file() or go2_file_sha(root/entry['file'])!=entry['sha256']:raise RuntimeError('Corrupt playing candidate; no fallback.')
    return manifest

def go2_teacher_preserve_candidate(project,run_id,which='latest',evaluation=None,*,_allow_smoke=False):
    project=go2_project(project);state,entry=go2_load_checkpoint(project/'checkpoints'/run_id,which=which)
    if state is None or state['global_step']<=0 or (state['config'].get('is_smoke') and not _allow_smoke):raise RuntimeError('Only real trained candidates can be published.')
    root=go2_teacher_candidate_root(project,run_id)
    with go2_checkpoint_lock(root):
        manifest=go2_teacher_candidate_manifest(project,run_id)
        entries=copy.deepcopy(manifest['entries']) if manifest else []
        if entry['file'] not in {item['file'] for item in entries}:
            go2_atomic_bytes(root/entry['file'],(project/'checkpoints'/run_id/entry['file']).read_bytes())
            if go2_file_sha(root/entry['file'])!=entry['sha256']:raise RuntimeError('Candidate copy hash mismatch.')
            entries.append({'file':entry['file'],'sha256':entry['sha256'],'optimizer_step':state['global_step'],
                'model_identity':state['identity'].get('model_identity'), 'training_identity':go2_digest(state['identity']),
                'evaluations':[]})
        candidate=next(item for item in entries if item['file']==entry['file'])
        if evaluation is not None:candidate['evaluations'].append({'eval_id':evaluation['eval_id'],'summary':evaluation['summary'],'config':evaluation['config']})
        best=manifest['best'] if manifest else entry['file']
        latest=max(entries,key=lambda item:item['optimizer_step'])['file']
        keep={item['file'] for item in sorted(entries,key=lambda item:item['optimizer_step'])[-2:]}|{best,entry['file']}
        updated={'schema':'ahr-v2-playing-candidates-v1','run_id':run_id,'entries':[item for item in entries if item['file'] in keep],
            'latest':latest,'best':best,'selection_basis':manifest.get('selection_basis','Awaiting independent playing comparison') if manifest else 'Awaiting independent playing comparison',
            'updated_at_utc':go2_now(),'resume_cell':'15: resume the saved pipeline; each candidate contains the complete original training state.'}
        go2_atomic_json(root/'manifest.json',updated)
        for item in entries:
            if item['file'] not in keep:(root/item['file']).unlink(missing_ok=True)
    return candidate

def go2_load_selected_teacher_player(project=None,run_id=None,which='best',device='cpu',candidate_file=None):
    project=go2_project(project)
    if run_id is None:run_id=json.loads((project/'reports/ahr_v2/teacher_active_run.json').read_text())['run_id']
    manifest=go2_teacher_candidate_manifest(project,run_id)
    namespace=go2_teacher_arena_namespace(project)
    if manifest is None:return namespace['go2_load_teacher_player'](project,run_id,which=which,device=device)
    name=candidate_file or manifest[which]
    if name not in {item['file'] for item in manifest['entries']}:raise RuntimeError('Requested playing snapshot is no longer retained.')
    entry=next(item for item in manifest['entries'] if item['file']==name)
    state=torch.load(go2_teacher_candidate_root(project,run_id)/name,map_location='cpu',weights_only=False)
    if GO2_REQUIRED_STATE-set(state) or state['schema']!=GO2_SCHEMA:raise RuntimeError('Incomplete candidate checkpoint.')
    if state['identity']['sources']!=namespace['go2_teacher_training_sources'](project):raise RuntimeError('Candidate training sources changed.')
    if go2_digest(state['identity'])!=entry['training_identity'] or state['global_step']!=entry['optimizer_step']:raise RuntimeError('Candidate metadata differs from original full checkpoint.')
    with torch.random.fork_rng(devices=[]):model=namespace['Go2Network'](namespace['Go2NetConfig'](board_size=9,**state['config']['network']))
    model.load_state_dict(state['model']);model.to(device);model.eval();model.requires_grad_(False)
    metadata={'run_id':run_id,'global_step':state['global_step'],'optimizer_step':state['global_step'],'model_identity':namespace['go2_model_identity'](model),
        'checkpoint_file':'playing_candidates/'+name,'checkpoint_sha256':entry['sha256'],'config':state['config'],'sources':state['identity']['sources'],
        'data_identity':state['identity']['data'],'training_source':state['config']['training_source'],'is_smoke':state['config'].get('is_smoke',False),
        'best_selection_basis':manifest['selection_basis'],'label':'Teacher-trained AHR playing candidate','development':state['metrics'].get('last_development')}
    return model,metadata

def go2_teacher_select_candidate(project,run_id,candidate_file,selection_basis):
    root=go2_teacher_candidate_root(project,run_id)
    with go2_checkpoint_lock(root):
        manifest=go2_teacher_candidate_manifest(project,run_id)
        if candidate_file not in {entry['file'] for entry in manifest['entries']}:raise RuntimeError('Cannot select an unretained candidate.')
        manifest.update(best=candidate_file,selection_basis=selection_basis,updated_at_utc=go2_now())
        go2_atomic_json(root/'manifest.json',manifest)
    return manifest


## Detached preparation, pretraining and independent reviews

In [ ]:
def go2_teacher_default_pipeline_config(project=None):
    project=go2_project(project);namespace=go2_teacher_arena_namespace(project)
    training=namespace['go2_teacher_training_config'](run_id='ahr-v2-teacher-20261008',cache_id='go9x9-teacher-v1',device='cuda:1',time_budget_hours=8)
    training['max_optimizer_steps']=int(np.ceil(500000/training['batch_size']))*training['max_epochs']
    return {'schema':'ahr-v2-teacher-pipeline-v1','run_id':training['run_id'],'training':training,
        'preparation':{'cache_id':training['cache_id'],'train_positions':500000,'dev_positions':8192,'max_positions_per_game':32,'games_per_commit':128,'seed':2026100812},
        'reviews':{'periodic_steps':[2000,8000],'periodic_games':4,'comparison_games':8,'confirmation_games':16,'higher_level_games':8,
            'simulations':128,'batch_size':8,'max_plies':600,'game_time_seconds':900.0,'move_time_seconds':None,'seed':2026100815},
        'sources':go2_teacher_control_sources(project),'test_consumed':False}

def go2_teacher_preflight(project,config):
    project=go2_project(project)
    if config.get('schema')!='ahr-v2-teacher-pipeline-v1' or config.get('test_consumed') is not False:raise ValueError('Invalid pipeline contract.')
    if config['sources']!=go2_teacher_control_sources(project):raise RuntimeError('Pinned pipeline sources differ; no restart.')
    namespace=go2_teacher_arena_namespace(project);namespace['go2_validate_teacher_training_config'](config['training'])
    path=project/'reports/ahr_v2/teacher_launch_validation.json'
    if not path.exists():raise RuntimeError('Complete the checkpoint/search/arena/UI preflight validation first.')
    receipt=json.loads(path.read_text())
    if receipt.get('status')!='PASS' or receipt.get('sources')!=config['sources']:raise RuntimeError('Validation evidence is absent or stale.')
    for artifact in receipt['artifacts']:
        if go2_file_sha(project/artifact['path'])!=artifact['sha256']:raise RuntimeError('Preflight validation artifact changed.')
    if shutil.disk_usage(project).free<12*1024**3:raise RuntimeError('Less than 12 GiB persistent storage free.')
    if not torch.cuda.is_available() or torch.cuda.device_count()<2:raise RuntimeError('GPU1 required for this saved configuration.')
    if not shutil.which('tmux') or not shutil.which('gnugo'):raise RuntimeError('Required tmux/GNU Go dependency unavailable.')
    return {'validated_at_utc':go2_now(),'configuration_sha256':go2_digest(config),'sources':config['sources'],'storage_free_bytes':shutil.disk_usage(project).free,'training_gpu':1,'evaluation_gpu':0}

def go2_teacher_pipeline(project,config):
    project=go2_project(project);run_id=config['run_id'];report_dir=project/'reports/ahr_v2'/run_id;report_dir.mkdir(parents=True,exist_ok=True)
    control_path=report_dir/'control.json';identity=go2_digest(config)
    if control_path.exists():
        control=json.loads(control_path.read_text())
        if control.get('identity')!=identity:raise RuntimeError('Pipeline control/config identity changed; no restart.')
    else:
        control={'schema':'ahr-v2-teacher-control-v1','identity':identity,'run_id':run_id,'stage':'PREPARING','next_periodic':0,'reviews':[],'pending_review':None,'final_comparison':[],'confirmation':None,'higher_levels':[]}
    def save_control():control['updated_at_utc']=go2_now();go2_atomic_json(control_path,control)
    save_control();namespace=go2_teacher_arena_namespace(project)
    def prepare_progress(status):
        go2_atomic_json(report_dir/'worker.json',{'status':'RUNNING','stage':'PREPARING','preparation':status,'updated_at_utc':go2_now(),'identity':identity})
    dataset=namespace['go2_prepare_teacher_cache'](project,**config['preparation'],on_commit=prepare_progress)
    control['data_summary']=namespace['go2_teacher_cache_summary'](dataset);control['data_identity']=dataset['identity'];save_control()
    def arena_config(games,level=1,seed=None):
        review=config['reviews']
        return {key:review[key] for key in ('simulations','batch_size','max_plies','game_time_seconds','move_time_seconds')}|{'games':games,'gnu_level':level,'seed':review['seed'] if seed is None else seed,'purpose':'development'}
    def run_review(record):
        control['stage']='REVIEWING';save_control()
        go2_atomic_json(report_dir/'worker.json',{'status':'RUNNING','stage':'REVIEWING','eval_id':record['eval_id'],'updated_at_utc':go2_now(),'identity':identity})
        arena=go2_teacher_arena_namespace(project)
        arena['go2_load_player']=lambda project,run_id=None,which='best',device='cpu':go2_load_selected_teacher_player(project,run_id or config['run_id'],device=device,candidate_file=record['candidate_file'])
        return go2_teacher_arena(project,run_id=run_id,eval_id=record['eval_id'],config=record['config'],device='cuda:0',namespace=arena)
    def finish_pending():
        if control['pending_review'] is None:return
        record=control['pending_review'];review=run_review(record)
        control['reviews'].append({'optimizer_step':record['optimizer_step'],'candidate_file':record['candidate_file'],'eval_id':review['eval_id'],'summary':review['summary'],'diagnostics':review['diagnostics']})
        # All periodic comparisons use the same development configuration. Keep
        # their best playing candidate independently of the training-loss best.
        complete=[item for item in control['reviews'] if item['summary']['unknown']==0]
        if complete:
            selected=max(complete,key=lambda item:(item['summary']['wins'],item['diagnostics']['mean_margin'] if item['diagnostics']['mean_margin'] is not None else -1000))
            retained=go2_teacher_candidate_manifest(project,run_id)
            if selected['candidate_file'] in {entry['file'] for entry in retained['entries']}:
                go2_teacher_select_candidate(project,run_id,selected['candidate_file'],'Periodic matched GNU Go level-1 development matches; small sample, awaiting final comparison')
        control['next_periodic']+=1;control['pending_review']=None;control['stage']='TRAINING';save_control()
        go2_atomic_json(report_dir/'worker.json',{'status':'RUNNING','stage':'TRAINING','updated_at_utc':go2_now(),'identity':identity})
    finish_pending()
    def on_checkpoint(state,entry):
        go2_atomic_json(report_dir/'worker.json',{'status':'RUNNING','stage':'TRAINING','optimizer_step':state['global_step'],'rows_seen':state['sampler']['rows_seen'],'updated_at_utc':go2_now(),'identity':identity})
        due=config['reviews']['periodic_steps'];index=control['next_periodic']
        if index>=len(due) or state['global_step']<due[index] or state['global_step']<=0:return
        candidate=go2_teacher_preserve_candidate(project,run_id,'latest')
        control['pending_review']={'optimizer_step':state['global_step'],'candidate_file':candidate['file'],
            'eval_id':run_id+'-periodic-dev-'+str(state['global_step']),'config':arena_config(config['reviews']['periodic_games'])}
        save_control();finish_pending()
    control['stage']='TRAINING';save_control()
    go2_atomic_json(report_dir/'worker.json',{'status':'RUNNING','stage':'TRAINING','updated_at_utc':go2_now(),'identity':identity})
    state=namespace['go2_run_teacher_training'](project,config['training'],on_checkpoint=on_checkpoint)
    training_status=json.loads((report_dir/'status.json').read_text())['status'];control['training_status']=training_status;save_control()
    if training_status=='STOPPED':
        control['stage']='STOPPED';save_control();return control
    control['stage']='FINAL_COMPARISON';save_control()
    # Preserve both final training candidates. The independently selected
    # periodic best is retained by the playing manifest as well.
    for choice in ('latest','best'):
        saved,_=go2_load_checkpoint(project/'checkpoints'/run_id,which=choice)
        if saved['global_step']>0:go2_teacher_preserve_candidate(project,run_id,choice)
        else:control['excluded_untrained_loss_best']=True
    candidates=go2_teacher_candidate_manifest(project,run_id)['entries']
    for candidate in list(candidates):
        eval_id=run_id+'-comparison-dev-'+str(candidate['optimizer_step'])
        if any(item['eval_id']==eval_id for item in control['final_comparison']):continue
        record={'candidate_file':candidate['file'],'eval_id':eval_id,'config':arena_config(config['reviews']['comparison_games'],seed=config['reviews']['seed']+1000)}
        review=run_review(record)
        control['final_comparison'].append({'candidate_file':candidate['file'],'optimizer_step':candidate['optimizer_step'],'eval_id':eval_id,'summary':review['summary'],'diagnostics':review['diagnostics']});save_control()
    eligible=[item for item in control['final_comparison'] if item['summary']['unknown']==0]
    if not eligible:raise RuntimeError('No complete candidate comparison; UNKNOWN cannot select a proved winner.')
    selected=max(eligible,key=lambda item:(item['summary']['wins'],item['diagnostics']['mean_margin'] if item['diagnostics']['mean_margin'] is not None else -1000))
    go2_teacher_select_candidate(project,run_id,selected['candidate_file'],'Matched 8-game GNU Go level-1 development comparison, equal 128-visit AHR budget; selection is not a final/lockbox result')
    control['selected']=selected;save_control()
    if control['confirmation'] is None:
        record={'candidate_file':selected['candidate_file'],'eval_id':run_id+'-selected-confirmation-dev','config':arena_config(config['reviews']['confirmation_games'],seed=config['reviews']['seed']+2000)}
        review=run_review(record);control['confirmation']={'eval_id':review['eval_id'],'summary':review['summary'],'diagnostics':review['diagnostics']};save_control()
    if control['confirmation']['summary']['observed_half_target']:
        for level in (5,10):
            if any(item['level']==level for item in control['higher_levels']):continue
            record={'candidate_file':selected['candidate_file'],'eval_id':run_id+'-gnu-level-'+str(level)+'-dev','config':arena_config(config['reviews']['higher_level_games'],level=level,seed=config['reviews']['seed']+3000+level*100)}
            review=run_review(record);control['higher_levels'].append({'level':level,'eval_id':review['eval_id'],'summary':review['summary'],'diagnostics':review['diagnostics']});save_control()
    control['stage']='COMPLETE';control['completed_at_utc']=go2_now();save_control()
    return control

def go2_teacher_worker(project,config_path,launch_path):
    project=go2_project(project);config=json.loads(Path(config_path).read_text());launch=json.loads(Path(launch_path).read_text())
    if go2_digest(config)!=launch['configuration_sha256'] or config['sources']!=go2_teacher_control_sources(project):raise RuntimeError('Worker source/config pin changed.')
    report=project/'reports/ahr_v2'/config['run_id']/'worker.json'
    try:
        result=go2_teacher_pipeline(project,config)
        go2_atomic_json(report,{'status':'RETURNED','stage':result['stage'],'returned_at_utc':go2_now(),'training_status':result.get('training_status'),'selected':result.get('selected'),'confirmation':result.get('confirmation'),'higher_levels':result.get('higher_levels'),'identity':go2_digest(config)})
        return result
    except BaseException as error:
        go2_atomic_json(report,{'status':'ERROR','error':repr(error),'updated_at_utc':go2_now(),'identity':go2_digest(config),'resume':'go2_launch_teacher(PROJECT,resume=True) with unchanged configuration and source exports'})
        raise

def go2_launch_teacher(project=None,config_path='config/ahr_v2_teacher_pipeline.json',session='ahr-v2-teacher',resume=False):
    project=go2_project(project);config_path=project/config_path;config=json.loads(config_path.read_text());preflight=go2_teacher_preflight(project,config)
    if subprocess.run(['tmux','has-session','-t',session],capture_output=True).returncode==0:raise RuntimeError('Session already exists; inspect/attach rather than launching another writer.')
    directory=project/'checkpoints'/config['run_id']
    if (directory/'manifest.json').exists() and not resume:raise RuntimeError('Training checkpoint exists; exact resume=True required.')
    report_dir=project/'reports/ahr_v2'/config['run_id'];report_dir.mkdir(parents=True,exist_ok=True)
    worker=report_dir/'detached_worker.ipynb';launch_path=report_dir/'launch.json';log=report_dir/'tmux.log'
    launch={**preflight,'run_id':config['run_id'],'session':session,'launched_at_utc':go2_now(),'resume':bool(resume),
        'config_path':str(config_path.relative_to(project)),'checkpoint_path':'checkpoints/'+config['run_id'],'play_notebook':GO2_ROOT+'/16_play_teacher_trained_player.ipynb',
        'attach_command':'tmux attach -t '+session,'detach_keys':'Ctrl-b, then d','log_path':str(log.relative_to(project))}
    lines=['from pathlib import Path','import nbformat','PROJECT=Path('+repr(str(project))+')',
        'for name in ["00_contracts_and_recovery.ipynb","15_teacher_evaluation_and_launch.ipynb"]:',
        '    for cell in nbformat.read(PROJECT/"experiments/ahr_v2"/name,4).cells:',
        '        if cell.cell_type=="code" and "ahr-export" in cell.metadata.get("tags",[]):exec(compile(cell.source,name+":ahr-export","exec"),globals())',
        'result=go2_teacher_worker(PROJECT,Path('+repr(str(config_path))+'),Path('+repr(str(launch_path))+'))',
        'print("Teacher training/review worker returned:",result["stage"])']
    book=nbformat.v4.new_notebook(metadata={'kernelspec':{'name':'ahr-go','language':'python','display_name':'AHR Go'}},cells=[nbformat.v4.new_markdown_cell('# Detached teacher-trained AHR worker — generated by notebook 15'),nbformat.v4.new_code_cell(chr(10).join(lines))])
    go2_atomic_bytes(worker,nbformat.writes(book).encode());go2_atomic_json(launch_path,launch)
    args=[str(project/'.venv/bin/python'),'-m','jupyter','nbconvert','--execute','--to','notebook','--inplace','--ExecutePreprocessor.kernel_name=ahr-go','--ExecutePreprocessor.timeout=-1',str(worker)]
    command=' '.join(shlex.quote(argument) for argument in args)+' >> '+shlex.quote(str(log))+' 2>&1'
    subprocess.run(['tmux','new-session','-d','-s',session,'-c',str(project),command],check=True,capture_output=True,text=True)
    go2_atomic_json(project/'reports/ahr_v2/teacher_active_run.json',launch);go2_atomic_json(project/'config/ahr_v2_teacher_active_run.json',launch)
    return launch


def go2_prepare_teacher_launch(project=None):
    """Record reviewable validation evidence and a frozen first-run configuration."""
    project=go2_project(project);sources=go2_teacher_control_sources(project)
    names=('teacher_data_validation.json','teacher_data_visual_qa.json','teacher_training_validation.json','teacher_production_recovery.json','teacher_training_visual_qa.json','worker_reading_validation.json','worker_reading_visual_qa.json','teacher_arena_validation.json','teacher_play_validation.json','teacher_control_visual_qa.json','teacher_play_visual_qa.json')
    artifacts=[]
    for name in names:
        path=project/'reports/ahr_v2'/name
        if not path.exists():raise RuntimeError('Missing validation artifact: '+name)
        record=json.loads(path.read_text())
        signal=record.get('status') or record.get('presentation_status')
        if signal not in ('PASS','PASSED') and record.get('all_passed') is not True:raise RuntimeError('Validation did not pass: '+name)
        recorded=record.get('source_identity',record.get('sources',{}))
        if isinstance(recorded,dict):
            for relative,digest in recorded.items():
                if relative in sources and sources[relative]!=digest:raise RuntimeError('Validation source is stale: '+name+' / '+relative)
        artifacts.append({'path':str(path.relative_to(project)),'sha256':go2_file_sha(path)})
    receipt={'status':'PASS','sources':sources,'artifacts':artifacts,'validated_at_utc':go2_now(),'scope':'Dataset, optimizer, exact GPU recovery, tactical search, arena recovery and human interface; no strength claim before matches.'}
    go2_atomic_json(project/'reports/ahr_v2/teacher_launch_validation.json',receipt)
    config=go2_teacher_default_pipeline_config(project)
    path=project/'config/ahr_v2_teacher_pipeline.json'
    if path.exists() and json.loads(path.read_text())!=config:raise RuntimeError('A different pipeline configuration already exists; do not overwrite a run identity.')
    go2_atomic_json(path,config);go2_atomic_json(project/'config/ahr_v2_teacher_training.json',config['training'])
    return {'config':config,'validation':receipt}


## Run the bounded arena check and inspect current progress

In [ ]:
ARENA_VALIDATION=go2_teacher_arena_validation(PROJECT)
print(ARENA_VALIDATION['status'],ARENA_VALIDATION['checks'])


In [ ]:
import pandas as pd
from IPython.display import display,HTML
pointer=PROJECT/'reports/ahr_v2/teacher_active_run.json'
if pointer.exists():
    active=json.loads(pointer.read_text());directory=PROJECT/'reports/ahr_v2'/active['run_id']
    for filename in ('worker.json','status.json','control.json'):
        path=directory/filename
        if path.exists():
            record=json.loads(path.read_text())
            display(HTML('<h3>'+filename+'</h3>'))
            display(pd.DataFrame([{'Field':key,'Value':str(record.get(key))} for key in ('status','stage','optimizer_step','rows_seen','active_seconds','training_status','updated_at_utc') if key in record]))
    control_path=directory/'control.json'
    if control_path.exists():
        control=json.loads(control_path.read_text())
        rows=[{'Step':item['optimizer_step'],'Wins':item['summary']['wins'],'Games':item['summary']['attempts'],'UNKNOWN':item['summary']['unknown'],'Mean margin':item['diagnostics']['mean_margin']} for item in control.get('reviews',[])]
        if rows:display(pd.DataFrame(rows))
else:display(HTML('<p>No main teacher run has been launched yet. The preflight checks above are correctness fixtures.</p>'))


In [ ]:
import matplotlib.pyplot as plt
if pointer.exists():
    active=json.loads(pointer.read_text());directory=PROJECT/'reports/ahr_v2'/active['run_id']
    train_path=directory/'metrics.jsonl';dev_path=directory/'development.jsonl'
    if train_path.exists() and dev_path.exists():
        train=pd.DataFrame([json.loads(line) for line in train_path.read_text().splitlines() if line.strip()])
        dev=pd.DataFrame([json.loads(line) for line in dev_path.read_text().splitlines() if line.strip()])
        if not train.empty and not dev.empty:
            plt.rcParams.update({'font.size':11,'axes.spines.top':False,'axes.spines.right':False})
            fig,axes=plt.subplots(1,2,figsize=(11,3.6),layout='constrained')
            axes[0].plot(train['optimizer_step'],train['loss'].ewm(span=100).mean(),label='Training loss, smoothed',color='#246c76')
            axes[0].plot(dev['optimizer_step'],dev['joint_loss'],'o-',label='Fixed development loss',color='#bb6d27')
            axes[0].set(title='Teacher learning on separate game families',xlabel='Optimizer updates',ylabel='Joint objective');axes[0].legend()
            axes[1].plot(dev['optimizer_step'],dev['policy_top1_agreement']*100,'o-',color='#246c76')
            axes[1].set(title='Unseen-family move agreement with teacher',xlabel='Optimizer updates',ylabel='Agreement (%)',ylim=(0,100))
            display(fig);plt.close(fig)
            display(HTML('<p>These curves measure learning from the teacher. Independent GNU Go matches determine observed playing strength.</p>'))


### Start or resume explicitly

After all five notebooks have passed their fresh-kernel checks and saved previews have been inspected, the launcher uses the pinned configuration under `config/`. It refuses changed or corrupt state.

```python
launch = go2_launch_teacher(PROJECT)                 # first launch
launch = go2_launch_teacher(PROJECT, resume=True)    # exact continuation after interruption
```

Live progress: `reports/ahr_v2/<run_id>/worker.json`, `status.json`, and `control.json`. A graceful training stop is requested by creating `checkpoints/<run_id>/STOP`; remove it only when explicitly resuming. Candidate strength is reported in independent matches rather than inferred from training loss.